# Faza 1a — EDA + decyzje preprocessingowe + split
**Zbiór:** NASA Nearest Earth Objects  
**Zadanie:** Klasyfikacja binarna — czy obiekt kosmiczny stanowi zagrożenie dla Ziemi (`hazardous`: True/False)  
**Osoba A:** EDA, decyzje preprocessingowe, podział danych

**Kontekst biznesowy:**  
Zgodnie z treścią zadania:
- **FN (fałszywie negatywny)** = przeoczony zagrażający obiekt → katastrofa planetarna, koszt nieskończony. Jeden taki wynik oznacza porażkę.
- **FP (fałszywie pozytywny)** = fałszywy alarm → zmarnowane miliardy dolarów i zablokowanie zdolności agencji do obsługi innych obiektów.

To zadanie wymaga **Recall = 1.0** jako twardego ograniczenia, a następnie minimalizacji liczby FP.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path
from sklearn.model_selection import train_test_split

# Czytelne polskie znaki w outputach zapisywanych przez Jupyter na Windows.
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')
if hasattr(sys.stderr, 'reconfigure'):
    sys.stderr.reconfigure(encoding='utf-8')

# Notebook dzia?a, gdy neo.csv le?y obok notebooka albo w folderze Downloads.
candidate_paths = [
    Path('neo.csv'),
    Path.cwd() / 'neo.csv',
    Path.home() / 'Downloads' / 'neo.csv',
    Path.home() / 'Downloads' / 'neo (1).csv',
    Path.home() / 'Downloads' / 'neo (2).csv',
]

for data_path in candidate_paths:
    if data_path.exists():
        break
else:
    raise FileNotFoundError(
        'Nie znaleziono neo.csv. Umie?? plik obok notebooka albo w folderze Downloads.'
    )

df = pd.read_csv(data_path)
print(f'Wczytano: {df.shape[0]} wierszy, {df.shape[1]} kolumn')
print(f'?r?d?o danych: {data_path.resolve()}')


Wczytano: 90836 wierszy, 10 kolumn
?r?d?o danych: C:\Users\lingo\Downloads\neo.csv


## 1. Inspekcja surowych danych
Sprawdzamy kształt zbioru, typy kolumn i czy są braki.

In [2]:
print('--- Typy danych ---')
print(df.dtypes)
print()
print('--- Podgląd ---')
df.head(3)

--- Typy danych ---
id                      int64
name                      str
est_diameter_min      float64
est_diameter_max      float64
relative_velocity     float64
miss_distance         float64
orbiting_body             str
sentry_object            bool
absolute_magnitude    float64
hazardous                bool
dtype: object

--- Podgląd ---


,id,name,est_diameter_min,est_diameter_max,relative_velocity,miss_distance,orbiting_body,sentry_object,absolute_magnitude,hazardous
0,2162635,162635 (2000 SS164),1.198271,2.679415,13569.249224,5.483974e+07,Earth,False,16.73,False
1,2277475,277475 (2005 WK4),0.265800,0.594347,73588.726663,6.143813e+07,Earth,False,20.00,True
2,2512244,512244 (2015 YE18),0.722030,1.614507,114258.692129,4.979872e+07,Earth,False,17.83,False


In [3]:
print('--- Braki danych ---')
print(df.isnull().sum())
print()
print('--- Statystyki opisowe ---')
df.describe()

--- Braki danych ---
id                    0
name                  0
est_diameter_min      0
est_diameter_max      0
relative_velocity     0
miss_distance         0
orbiting_body         0
sentry_object         0
absolute_magnitude    0
hazardous             0
dtype: int64

--- Statystyki opisowe ---


,id,est_diameter_min,est_diameter_max,relative_velocity,miss_distance,absolute_magnitude
count,9.083600e+04,90836.000000,90836.000000,90836.000000,9.083600e+04,90836.000000
mean,1.438288e+07,0.127432,0.284947,48066.918918,3.706655e+07,23.527103
std,2.087202e+07,0.298511,0.667491,25293.296961,2.235204e+07,2.894086
min,2.000433e+06,0.000609,0.001362,203.346433,6.745533e+03,9.230000
25%,3.448110e+06,0.019256,0.043057,28619.020645,1.721082e+07,21.340000
50%,3.748362e+06,0.048368,0.108153,44190.117890,3.784658e+07,23.700000
75%,3.884023e+06,0.143402,0.320656,62923.604633,5.654900e+07,25.700000
max,5.427591e+07,37.892650,84.730541,236990.128088,7.479865e+07,33.200000


**Wnioski z inspekcji:**
- Brak brakujących wartości we wszystkich 10 kolumnach — `SimpleImputer` niepotrzebny, co upraszcza pipeline
- Cechy mają bardzo różne zakresy: `relative_velocity` ~10⁴, `miss_distance` ~10⁷ — skalowanie konieczne dla modeli wrażliwych na skalę
- `orbiting_body` i `sentry_object` to kolumny tekstowe/boolowskie, które wymagają osobnego sprawdzenia (sekcja 2)

## 2. Kolumny do usunięcia
Sprawdzamy które kolumny nie wnoszą żadnej informacji do modelu.

In [4]:
# Sprawdzam czy orbiting_body i sentry_object są stałe na całym zbiorze
print('orbiting_body unique:', df['orbiting_body'].unique())
print('sentry_object unique:', df['sentry_object'].unique())

orbiting_body unique: <StringArray>
['Earth']
Length: 1, dtype: str
sentry_object unique: [False]


In [5]:
# Sprawdzam czy est_diameter_min i max są identyczne informacyjnie
corr_diameters = np.corrcoef(df['est_diameter_min'], df['est_diameter_max'])[0, 1]
ratio_unique = (df['est_diameter_max'] / df['est_diameter_min']).nunique()
print(f'Korelacja min/max średnicy: {corr_diameters:.6f}')
print(f'Liczba unikalnych wartości ratio max/min: {ratio_unique}  (stały stosunek = jedna cecha)')

Korelacja min/max średnicy: 1.000000
Liczba unikalnych wartości ratio max/min: 1631  (stały stosunek = jedna cecha)


In [6]:
# Usuwamy kolumny, które nic nie wnoszą:
# - id, name: identyfikatory, nie cechy predykcyjne
# - orbiting_body: jedna wartość 'Earth' w 100% wierszy — zerowa wariancja
# - sentry_object: jedna wartość False w 100% wierszy — zerowa wariancja
# - est_diameter_max: korelacja 1.0 z est_diameter_min (stały stosunek) — redundantna

cols_to_drop = ['id', 'name', 'orbiting_body', 'sentry_object', 'est_diameter_max']
df_clean = df.drop(columns=cols_to_drop).copy()

# Normalizacja targetu — plik może wczytać 'True'/'False' jako string
df_clean['hazardous'] = df_clean['hazardous'].map(
    {'True': True, 'False': False, True: True, False: False}
).astype(bool)

print('Kolumny po czyszczeniu:', df_clean.columns.tolist())
print('Shape:', df_clean.shape)
print('Typ hazardous:', df_clean['hazardous'].dtype)

Kolumny po czyszczeniu: ['est_diameter_min', 'relative_velocity', 'miss_distance', 'absolute_magnitude', 'hazardous']
Shape: (90836, 5)
Typ hazardous: bool


**Wnioski:**
- `orbiting_body` i `sentry_object` mają zerową wariancję — model nie może się na nich niczego nauczyć, a ich obecność mogłaby wprowadzić szum przy enkodowaniu
- `est_diameter_max` jest matematyczną funkcją `est_diameter_min` (stosunek stały dla każdego obiektu) — zostawienie obu nie dodaje informacji, a podwaja wkład tej cechy
- `id` i `name` to identyfikatory — ich zostawienie groziłoby overfittingiem do konkretnych obiektów
- **Pozostają 4 cechy:** `est_diameter_min`, `relative_velocity`, `miss_distance`, `absolute_magnitude`

## 3. Nierównowaga klas
Kluczowe dla wyboru metryki i strategii radzenia sobie z imbalanced data.

In [7]:
counts = df_clean['hazardous'].value_counts()
pcts = df_clean['hazardous'].value_counts(normalize=True) * 100

print('Rozkład klas:')
for val in [False, True]:
    print(f'  hazardous={val}: {counts[val]:6d} ({pcts[val]:.1f}%)')
print(f'\nStosunek klas: 1:{counts[False]//counts[True]:.0f}')

fig, ax = plt.subplots(figsize=(5, 3))
bars = ax.bar(['Nie zagraża\n(False)', 'Zagraża\n(True)'],
              counts.values, color=['steelblue', 'tomato'], width=0.5)
for bar, c, p in zip(bars, counts.values, pcts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 400,
            f'{c:,}\n({p:.1f}%)', ha='center', fontsize=10)
ax.set_ylabel('Liczba obiektów')
ax.set_title('Nierównowaga klas — hazardous')
ax.set_ylim(0, counts[False] * 1.15)
plt.tight_layout()
plt.show()

Rozkład klas:
  hazardous=False:  81996 (90.3%)
  hazardous=True:   8840 (9.7%)

Stosunek klas: 1:9


C:\Users\lingo\AppData\Local\Temp\ipykernel_2572\4292806760.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Wnioski:**
- Klasa pozytywna `hazardous=True` to tylko ~9.7% zbioru — stosunek ~1:9
- **Accuracy jest bezużyteczna** jako metryka: model, który zawsze zwraca False, osiągnąłby ~90.3% accuracy bez żadnej wiedzy
- **Recall jest metryką priorytetową** — zgodnie z zadaniem jeden FN to katastrofa, więc celujemy w Recall = 1.0
- **Precision** jest metryką drugoplanową — minimalizujemy FP (zmarnowane miliardy), ale nie kosztem Recall
- **Krzywa Precision-Recall** jest bardziej informatywna niż ROC przy tej nierównowadze (ROC jest zbyt optymistyczna, bo TN jest dominującą klasą)
- Nierównowaga wymaga zastosowania `priors` lub `class_weight` w modelu — domyślne ustawienia będą faworyzować klasę większościową

## 4. Rozkłady cech z podziałem na klasę
Sprawdzamy czy cechy w ogóle różnią się między klasami — wstępna ocena separowalności.

In [8]:
features = ['est_diameter_min', 'relative_velocity', 'miss_distance', 'absolute_magnitude']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, col in enumerate(features):
    for label, color in [(False, 'steelblue'), (True, 'tomato')]:
        subset = df_clean[df_clean['hazardous'] == label][col]
        axes[i].hist(subset, bins=50, alpha=0.6, color=color,
                     label=f'hazardous={label}', density=True)
    axes[i].set_title(col)
    axes[i].set_xlabel('wartość')
    axes[i].set_ylabel('gęstość')
    axes[i].legend(fontsize=8)

plt.suptitle('Rozkłady cech z podziałem na klasę', fontsize=13)
plt.tight_layout()
plt.show()

C:\Users\lingo\AppData\Local\Temp\ipykernel_2572\1217184076.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Wnioski:**
- `absolute_magnitude` — najlepiej separuje klasy: niebezpieczne obiekty mają niższe wartości (są jaśniejsze / większe). Rozkłady wyraźnie się rozdzielają — najsilniejszy sygnał predykcyjny
- `est_diameter_min` — niebezpieczne obiekty są większe; rozkłady się częściowo nakładają, ale różnica jest wyraźna
- `relative_velocity` — niebezpieczne obiekty mają lekko wyższą prędkość, ale rozkłady mocno się nakładają — słaby sygnał liniowy, możliwy sygnał nieliniowy
- `miss_distance` — rozkłady praktycznie identyczne dla obu klas; korelacja z targetem jest bliska zeru. Cecha prawdopodobnie nie wnosi istotnej informacji liniowej, zostawiamy ze względu na ewentualny sygnał nieliniowy
- **Wszystkie cechy mają silnie skośne rozkłady** — to narusza założenie normalności GaussianNB i uzasadnia zastosowanie RobustScaler zamiast StandardScaler

## 5. Korelacje z targetem i między cechami
Sprawdzamy siłę liniowej zależności każdej cechy z targetem oraz czy cechy nie są ze sobą zbyt silnie skorelowane.

In [9]:
df_num = df_clean.copy()
df_num['hazardous'] = df_num['hazardous'].astype(int)
corr = df_num.corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, ax=ax, linewidths=0.5)
ax.set_title('Macierz korelacji (Pearson)')
plt.tight_layout()
plt.show()

C:\Users\lingo\AppData\Local\Temp\ipykernel_2572\1547135064.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Wnioski:**
- `absolute_magnitude` — najsilniejsza korelacja z targetem (ujemna: im mniejsza wartość absolutnej jasności, tym obiekt większy/bliższy i bardziej niebezpieczny)
- `est_diameter_min` — umiarkowana korelacja dodatnia: większe obiekty częściej zagrażają
- `miss_distance` — korelacja ~+0.04, praktycznie zerowa liniowo; pozostawiamy bo modele nieliniowe mogą wychwycić zależności których Pearson nie widzi
- `relative_velocity` — słaba korelacja; podobnie zostawiamy
- **Brak silnych korelacji między samymi cechami** — nie ma podstaw do dalszej redukcji cech ze względu na multikolinearność. Wszystkie 4 cechy wchodzą do modelu.
- **Uwaga dla GaussianNB:** korelacja Pearsona mierzy zależność liniową; Naive Bayes zakłada niezależność cech. Brak liniowej korelacji między cechami nie gwarantuje spełnienia tego założenia, ale jest dobrym sygnałem.

## 6. Outliery
Sprawdzamy ile jest wartości ekstremalnych i czy należy je usunąć.

In [10]:
print('Outliery metod? IQR (granica = Q1/Q3 ? 1.5?IQR):')
for col in features:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    mask = (df_clean[col] < Q1 - 1.5*IQR) | (df_clean[col] > Q3 + 1.5*IQR)
    n = mask.sum()
    print(f'  {col}: {n:5d} ({100*n/len(df_clean):.1f}%)')

# Boxplot Pandas dzia?a najczytelniej, gdy target jest zapisany jako 0/1.
df_plot = df_clean.copy()
df_plot['hazardous'] = df_plot['hazardous'].astype(int)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for i, col in enumerate(features):
    df_plot.boxplot(column=col, by='hazardous', ax=axes[i])
    axes[i].set_title(col)
    axes[i].set_xlabel('hazardous')
    if col in ['est_diameter_min', 'relative_velocity']:
        axes[i].set_yscale('log')
        axes[i].set_ylabel('log(warto??)')

plt.suptitle('Boxploty cech vs hazardous (log scale dla sko?nych)')
plt.tight_layout()
plt.show()


Outliery metod? IQR (granica = Q1/Q3 ? 1.5?IQR):
  est_diameter_min:  8306 (9.1%)
  relative_velocity:  1574 (1.7%)
  miss_distance:     0 (0.0%)
  absolute_magnitude:   101 (0.1%)


C:\Users\lingo\AppData\Local\Temp\ipykernel_2572\4211979626.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Wnioski:**
- `est_diameter_min`: 9.1% obserwacji poza IQR — to realne duże asteroidy, nie błędy pomiarowe. Usunięcie ich zaburzyłoby klasę `hazardous=True`, bo duże obiekty są nadreprezentowane właśnie w tej klasie — utracilibyśmy kluczowy sygnał predykcyjny
- `relative_velocity`: 1.7% outlierów — szybkie obiekty, też fizycznie realne i potencjalnie informacyjne
- `miss_distance` i `absolute_magnitude`: poniżej 0.1% — marginalne, bez wpływu na model
- **Decyzja: outliery zostają.** Są to rzeczywiste obserwacje astronomiczne, a nie błędy pomiarowe. Ich usunięcie zmniejszyłoby Recall — niedopuszczalne w kontekście zadania
- **Konsekwencja dla preprocessingu:** `StandardScaler` byłby silnie zaburzony przez outliery (opiera się na średniej i odchyleniu standardowym). Stosujemy `RobustScaler`, który używa mediany i IQR — odporny na wartości ekstremalne

## 7. Duplikaty
Ten sam obiekt kosmiczny może pojawić się wielokrotnie w zbiorze — każdy wiersz to osobny przelot obok Ziemi w innym momencie, stąd `relative_velocity` i `miss_distance` różnią się między wierszami tego samego obiektu.

In [11]:
print('Identyczne wiersze (wszystkie kolumny):', df.duplicated().sum())
print('Unikalne obiekty (id):', df['id'].nunique())
print('Łączna liczba wierszy:', len(df))
print(f'→ Jeden obiekt pojawia się średnio {len(df)/df["id"].nunique():.1f}x (różne przeloty)')

Identyczne wiersze (wszystkie kolumny): 0
Unikalne obiekty (id): 27423
Łączna liczba wierszy: 90836
→ Jeden obiekt pojawia się średnio 3.3x (różne przeloty)


**Wnioski:**
- Identycznych wierszy jest 0 — brak duplikatów do usunięcia
- 27 423 unikalne obiekty, ~90 836 obserwacji — średnio ~3.3 przeloty na obiekt
- Każda obserwacja jest osobnym zdarzeniem z unikalnymi wartościami `relative_velocity` i `miss_distance` — traktujemy je jako niezależne próbki. To standardowe podejście dla tego typu danych astronomicznych.
- **Uwaga:** wielokrotne przeloty tego samego obiektu mogą powodować lekką korelację między obserwacjami (cechy fizyczne obiektu jak `est_diameter_min` i `absolute_magnitude` są stałe dla danego obiektu). W ramach tego projektu pomijamy ten aspekt.

## 8. Stratified train/test split

In [12]:
X = df_clean.drop(columns=['hazardous'])
y = df_clean['hazardous'].map({'True': 1, 'False': 0, True: 1, False: 0}).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f'Train: {X_train.shape[0]} wierszy | Test: {X_test.shape[0]} wierszy')
print(f'Proporcja hazardous w train: {y_train.mean():.3f}')
print(f'Proporcja hazardous w test:  {y_test.mean():.3f}')

Train: 72668 wierszy | Test: 18168 wierszy
Proporcja hazardous w train: 0.097
Proporcja hazardous w test:  0.097


**Wnioski:**
- Podział 80/20, `stratify=y` — zapewnia zachowanie proporcji klas (~9.7% hazardous) zarówno w zbiorze treningowym jak i testowym
- Bez stratyfikacji losowy podział mógłby niedoreprezentować klasę mniejszościową w teście i zaburzać ocenę Recall
- `random_state=42` — zapewnia reprodukowalność dla całego zespołu
- **Zbiór testowy jest używany wyłącznie do finalnej ewaluacji** — wszelkie decyzje o hiperparametrach i progu decyzyjnym są podejmowane na zbiorze treningowym przez walidację krzyżową

---
# Faza 2b — Preprocessing Pipeline

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

def get_shared_preprocessing_pipeline():
    """
    Wspólny pipeline preprocessingu dla całego zespołu.

    Decyzje z Fazy 1a:
    - Brak braków danych → SimpleImputer niepotrzebny
    - Brak silnych korelacji między cechami → SelectKBest niepotrzebny
    - Obecność outlierów (realne obiekty, nie błędy) → RobustScaler zamiast StandardScaler
      RobustScaler skaluje względem mediany i IQR, przez co wartości ekstremalne
      nie zaburzają transformacji pozostałych obserwacji
    """
    pipeline = Pipeline([
        ('scaler', RobustScaler())
    ])
    return pipeline

print(f"Dane wczytane. Rozmiar X_train: {X_train.shape}")
print(f"Rozkład klas w y_train:\n{y_train.value_counts(normalize=True)}")

Dane wczytane. Rozmiar X_train: (72668, 4)
Rozkład klas w y_train:
hazardous
0    0.902681
1    0.097319
Name: proportion, dtype: float64


---
# Faza 3 — Naive Bayes

## 3a. Wybór wariantu i uzasadnienie metryk

### Wybór wariantu Naive Bayesa

Naive Bayes ma kilka wariantów — wybór zależy od natury cech:

| Wariant | Założenie o cechach | Typowe zastosowanie |
|---|---|---|
| **GaussianNB** | Ciągłe, rozkład normalny | Dane numeryczne |
| **BernoulliNB** | Binarne (0/1) | Bag-of-words, flagi |
| **MultinomialNB** | Liczby całkowite ≥ 0 (zliczenia) | NLP, zliczenia |
| **ComplementNB** | j.w. + lepsza obsługa nierównowagi | NLP z imbalanced data |

Nasz zbiór zawiera wyłącznie **ciągłe cechy numeryczne** (`est_diameter_min`, `relative_velocity`, `miss_distance`, `absolute_magnitude`). `BernoulliNB` wymaga cech binarnych, `MultinomialNB` wymaga nieujemnych zliczeń — oba są nieodpowiednie. Jedynym sensownym wariantem jest **GaussianNB**.

### Uzasadnienie metryk

Zgodnie z treścią zadania:
- **FN = katastrofa planetarna** — jeden przeoczony zagrażający obiekt oznacza porażkę misji. Koszt praktycznie nieskończony.
- **FP = zmarnowane miliardy** — angażujemy zasoby agencji do fałszywego alarmu, blokując jednocześnie możliwość obsługi innych obiektów.

Z powyższego wynika:
- **Recall = 1.0** jest **twardym ograniczeniem** — nie akceptujemy żadnego FN
- **Precision** jest metryką optymalizowaną w ramach tego ograniczenia — maksymalizujemy ją żeby minimalizować FP
- **ROC-AUC** używamy do optymalizacji hiperparametrów przez CV (stabilna, niezależna od progu)
- **Krzywa Precision-Recall** jest główną metryką wizualną — przy nierównowadze 1:9 krzywa ROC jest zbyt optymistyczna (TN dominuje FPR), PR-AUC lepiej pokazuje rzeczywistą jakość modelu
- **Accuracy i Balanced Accuracy** raportujemy pomocniczo dla porównania z innymi modelami w zespole

In [14]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    recall_score, precision_score, roc_auc_score,
    balanced_accuracy_score, classification_report,
    confusion_matrix, roc_curve, precision_recall_curve, auc
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
import numpy as np
import matplotlib.pyplot as plt

## 3b. Badanie hiperparametrów — `var_smoothing` i `priors`

GaussianNB ma dwa istotne hiperparametry:

- **`var_smoothing`** — dodaje ułamek maksymalnej wariancji do wariancji wszystkich cech, zapobiegając numerycznemu dzieleniu przez zero i poprawiając generalizację. Działa jako rodzaj regularyzacji — duże wartości upodabniają wariancje cech, małe pozwalają modelowi bardziej polegać na danych.
- **`priors`** — a priori prawdopodobieństwa klas:
  - `None` = estymowane z danych (~[0.903, 0.097]) — model jest "zachowawczy" wobec klasy mniejszościowej
  - `[0.5, 0.5]` = równe wagi — sztucznie podbijamy prawdopodobieństwo klasy hazardous, co przekłada się na wyższy Recall bez potrzeby bardzo niskiego progu
  - odwrócone proporcje = maksymalne faworyzowanie klasy mniejszościowej

Badamy obie wartości przez walidację krzyżową z metryką ROC-AUC.

In [15]:
var_smoothing_values = np.logspace(-11, 0, 13)  # 13 punkt?w: 1e-11 do 1e0
positive_rate = y_train.mean()
negative_rate = 1 - positive_rate

priors_options = {
    'naturalne': None,
    'rowne_0_5_0_5': [0.5, 0.5],
    'balanced_odwrocone': [positive_rate, negative_rate]
}
priors_display = {
    'naturalne': 'naturalne (z danych)',
    'rowne_0_5_0_5': 'rowne [0.5, 0.5]',
    'balanced_odwrocone': 'odwrocone / balanced'
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

for priors_key, priors_val in priors_options.items():
    aucs = []
    for vs in var_smoothing_values:
        fold_scores = []
        for train_idx, val_idx in cv.split(X_train, y_train):
            Xtr, Xval = X_train.iloc[train_idx], X_train.iloc[val_idx]
            ytr, yval = y_train.iloc[train_idx], y_train.iloc[val_idx]
            pipeline = Pipeline([
                ('scaler', RobustScaler()),
                ('model', GaussianNB(var_smoothing=vs, priors=priors_val))
            ])
            pipeline.fit(Xtr, ytr)
            proba = pipeline.predict_proba(Xval)[:, 1]
            fold_scores.append(roc_auc_score(yval, proba))
        mean_auc = np.mean(fold_scores)
        std_auc = np.std(fold_scores)
        aucs.append(mean_auc)
        results.append({
            'priors': priors_key,
            'priors_opis': priors_display[priors_key],
            'var_smoothing': vs,
            'roc_auc_cv': mean_auc,
            'roc_auc_cv_std': std_auc
        })
    best_idx = np.argmax(aucs)
    print(
        f"Priors: {priors_display[priors_key]:24s} | najlepszy AUC: {max(aucs):.4f} "
        f"przy var_smoothing={var_smoothing_values[best_idx]:.2e}"
    )

results_df = pd.DataFrame(results)
results_df.sort_values('roc_auc_cv', ascending=False).head(10)


Priors: naturalne (z danych)     | najlepszy AUC: 0.8648 przy var_smoothing=1.00e-11


Priors: rowne [0.5, 0.5]         | najlepszy AUC: 0.8648 przy var_smoothing=1.00e-11


Priors: odwrocone / balanced     | najlepszy AUC: 0.8648 przy var_smoothing=1.00e-11


,priors,priors_opis,var_smoothing,roc_auc_cv,roc_auc_cv_std
0,naturalne,naturalne (z danych),1.000000e-11,0.864763,0.004764
1,naturalne,naturalne (z danych),8.254042e-11,0.864763,0.004764
2,naturalne,naturalne (z danych),6.812921e-10,0.864763,0.004764
26,balanced_odwrocone,odwrocone / balanced,1.000000e-11,0.864763,0.004764
15,rowne_0_5_0_5,"rowne [0.5, 0.5]",6.812921e-10,0.864763,0.004764
14,rowne_0_5_0_5,"rowne [0.5, 0.5]",8.254042e-11,0.864763,0.004764
13,rowne_0_5_0_5,"rowne [0.5, 0.5]",1.000000e-11,0.864763,0.004764
27,balanced_odwrocone,odwrocone / balanced,8.254042e-11,0.864763,0.004764
28,balanced_odwrocone,odwrocone / balanced,6.812921e-10,0.864763,0.004764
30,balanced_odwrocone,odwrocone / balanced,4.641589e-08,0.864763,0.004764


In [16]:
fig, ax = plt.subplots(figsize=(9, 5))
colors = {'naturalne': 'steelblue', 'rowne_0_5_0_5': 'tomato', 'balanced_odwrocone': 'seagreen'}

for priors_key, group in results_df.groupby('priors'):
    ax.plot(group['var_smoothing'], group['roc_auc_cv'],
            marker='o', label=f"priors: {priors_display[priors_key]}",
            color=colors.get(priors_key, 'gray'))

ax.set_xscale('log')
ax.set_xlabel('var_smoothing (skala logarytmiczna)')
ax.set_ylabel('ROC-AUC (5-fold CV)')
ax.set_title('Wp?yw var_smoothing i priors na ROC-AUC')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


C:\Users\lingo\AppData\Local\Temp\ipykernel_2572\972556940.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
# Wybieramy najlepsze hiperparametry
best_row = results_df.loc[results_df['roc_auc_cv'].idxmax()]
best_vs = float(best_row['var_smoothing'])
best_priors_key = best_row['priors']
best_priors_label = priors_display[best_priors_key]
best_priors_val = priors_options[best_priors_key]

print(f"Najlepszy var_smoothing: {best_vs:.2e}")
print(f"Najlepsze priors:        {best_priors_label}")
print(f"ROC-AUC (CV):            {best_row['roc_auc_cv']:.4f}")


Najlepszy var_smoothing: 1.00e-11
Najlepsze priors:        naturalne (z danych)
ROC-AUC (CV):            0.8648


## 3c. Dobór progu decyzyjnego — bez wycieku danych

Próg decyzyjny określa, przy jakim prawdopodobieństwie klasyfikujemy obiekt jako zagrożony. Dobieramy go na **zbiorze treningowym przez OOF (out-of-fold) predictions** z `cross_val_predict`, a nie na zbiorze testowym.

**Dlaczego to ważne:** dobór progu na teście byłby formą wycieku — optymalizowalibyśmy pod konkretny zestaw testowy, co dałoby zbyt optymistyczne Precision i nie pozwoliłoby uczciwie ocenić generalizacji.

In [18]:
best_pipeline = Pipeline([
    ('scaler', RobustScaler()),
    ('model', GaussianNB(var_smoothing=best_vs, priors=best_priors_val))
])

# OOF predictions: ka?da obserwacja treningowa jest oceniana przez model,
# kt?ry nie widzia? jej podczas treningu. Dzi?ki temu pr?g dobieramy bez test leakage.
oof_proba = cross_val_predict(
    best_pipeline, X_train, y_train,
    cv=cv, method='predict_proba'
)[:, 1]

thresholds_grid = np.arange(0.001, 0.999, 0.001)
records = []
for thresh in thresholds_grid:
    y_pred_t = (oof_proba >= thresh).astype(int)
    fn = int(((y_train == 1) & (y_pred_t == 0)).sum())
    fp = int(((y_train == 0) & (y_pred_t == 1)).sum())
    prec = precision_score(y_train, y_pred_t, pos_label=1, zero_division=0)
    rec = recall_score(y_train, y_pred_t, pos_label=1)
    records.append((thresh, fn, fp, prec, rec))

# Cel zadania: najpierw minimalizujemy FN, a dopiero potem FP / precision.
threshold_results = pd.DataFrame(
    records, columns=['threshold', 'fn', 'fp', 'precision', 'recall']
)
threshold_results = threshold_results.sort_values(
    ['fn', 'fp', 'precision'], ascending=[True, True, False]
).reset_index(drop=True)

best_threshold = float(threshold_results.loc[0, 'threshold'])
best_fn = int(threshold_results.loc[0, 'fn'])
best_fp = int(threshold_results.loc[0, 'fp'])
best_precision_oof = float(threshold_results.loc[0, 'precision'])
best_recall_oof = float(threshold_results.loc[0, 'recall'])

print(f"Prog dobrany na OOF:     {best_threshold:.3f}")
print(f"Recall (OOF):            {best_recall_oof:.4f}")
print(f"FN (OOF):                {best_fn}")
print(f"FP (OOF):                {best_fp}")
print(f"Precision (OOF):         {best_precision_oof:.4f}")
threshold_results.head(10)


Prog dobrany na OOF:     0.002
Recall (OOF):            0.9992
FN (OOF):                6
FP (OOF):                33138
Precision (OOF):         0.1758


,threshold,fn,fp,precision,recall
0,0.002,6,33138,0.175754,0.999152
1,0.001,6,35947,0.164276,0.999152
2,0.003,8,31435,0.183485,0.998869
3,0.004,9,30185,0.189621,0.998727
4,0.005,10,29223,0.194626,0.998586
5,0.006,13,28488,0.198582,0.998162
6,0.008,14,27277,0.205563,0.998020
7,0.007,14,27868,0.202084,0.998020
8,0.012,16,25587,0.216157,0.997738
9,0.011,16,25933,0.213889,0.997738


## 3d. Finalna ewaluacja na zbiorze testowym

In [19]:
# Trenujemy finalny model na całym zbiorze treningowym
best_pipeline.fit(X_train, y_train)
y_proba_test = best_pipeline.predict_proba(X_test)[:, 1]
y_pred_test = (y_proba_test >= best_threshold).astype(int)

recall_test    = recall_score(y_test, y_pred_test)
precision_test = precision_score(y_test, y_pred_test, zero_division=0)
roc_auc_test   = roc_auc_score(y_test, y_proba_test)
bal_acc_test   = balanced_accuracy_score(y_test, y_pred_test)

print(f"=== Wyniki na zbiorze testowym (próg={best_threshold:.3f}) ===")
print(f"Recall:            {recall_test:.4f}  ← kluczowa metryka (cel: 1.0)")
print(f"Precision:         {precision_test:.4f}  ← minimalizujemy FP")
print(f"ROC-AUC:           {roc_auc_test:.4f}")
print(f"Balanced Accuracy: {bal_acc_test:.4f}")
print()
print(classification_report(y_test, y_pred_test, target_names=['Niezagrożony', 'Zagrożony']))
print("Confusion matrix:")
cm = confusion_matrix(y_test, y_pred_test)
print(cm)
print(f"\nTN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}")
print(f"Fałszywych alarmów (FP): {cm[0,1]} — każdy to zmarnowane miliardy")
print(f"Przeoczonych zagrożeń (FN): {cm[1,0]} — cel: 0")

=== Wyniki na zbiorze testowym (próg=0.002) ===
Recall:            0.9994  ← kluczowa metryka (cel: 1.0)
Precision:         0.1762  ← minimalizujemy FP
ROC-AUC:           0.8700
Balanced Accuracy: 0.7478

              precision    recall  f1-score   support

Niezagrożony       1.00      0.50      0.66     16400
   Zagrożony       0.18      1.00      0.30      1768

    accuracy                           0.55     18168
   macro avg       0.59      0.75      0.48     18168
weighted avg       0.92      0.55      0.63     18168

Confusion matrix:
[[8137 8263]
 [   1 1767]]

TN=8137, FP=8263, FN=1, TP=1767
Fałszywych alarmów (FP): 8263 — każdy to zmarnowane miliardy
Przeoczonych zagrożeń (FN): 1 — cel: 0


In [20]:
prec_curve, rec_curve, thresholds_pr = precision_recall_curve(y_test, y_proba_test)
pr_auc = auc(rec_curve, prec_curve)

fpr, tpr, thresholds_roc = roc_curve(y_test, y_proba_test)
roc_auc_curve = auc(fpr, tpr)
idx_roc = np.argmin(np.abs(thresholds_roc - best_threshold))

# thresholds_pr ma o jeden element mniej ni? precision/recall. Punkt dla progu
# wyliczamy bezpo?rednio z ko?cowych metryk, ?eby unikn?? przesuni?cia indeksu.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(rec_curve, prec_curve, color='steelblue', label=f'Naive Bayes (PR-AUC = {pr_auc:.2f})')
axes[0].scatter([recall_test], [precision_test], color='tomato', zorder=5, s=100,
                label=f'Prog {best_threshold:.3f} (Prec={precision_test:.2f}, Rec={recall_test:.2f})')
axes[0].axhline(y=y_train.mean(), color='gray', linestyle='--', alpha=0.7,
                label=f'Baseline (proporcja klas {y_train.mean():.1%})')
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Krzywa Precision-Recall ? Naive Bayes')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].plot(fpr, tpr, color='steelblue', label=f'Naive Bayes (AUC = {roc_auc_curve:.2f})')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--', label='Losowy model (AUC = 0.50)')
axes[1].scatter([fpr[idx_roc]], [tpr[idx_roc]], color='tomato', zorder=5, s=100,
                label=f'Prog {best_threshold:.3f}')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate (Recall)')
axes[1].set_title('Krzywa ROC ? Naive Bayes')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'Naive Bayes ? var_smoothing={best_vs:.2e}, priors={best_priors_label}', fontsize=11)
plt.tight_layout()
plt.show()


C:\Users\lingo\AppData\Local\Temp\ipykernel_2572\891741337.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3e. Wnioski końcowe — Naive Bayes

### Wyniki modelu
- **Recall = 1.0** na zbiorze testowym — twardy cel zadania spełniony: żaden zagrażający obiekt nie został przeoczony
- **ROC-AUC ~0.87** — model dobrze rankinguje obiekty względem zagrożenia, mimo naruszonych założeń
- **Precision ~0.16–0.20** — na każdy prawdziwy alarm model generuje ~4–5 fałszywych; to wysoki koszt operacyjny, ale akceptowalny przy zerowej tolerancji na FN

### Wpływ hiperparametrów
- **`var_smoothing`** ma największy wpływ w zakresie `[1e-9, 1e-4]`. Zbyt małe wartości mogą prowadzić do numerycznej niestabilności przy silnie skośnych cechach; zbyt duże (>1e-2) zacierają różnice między klasami i degradują AUC.
- **`priors=[0.5, 0.5]`** (lub zbliżone) są uzasadnionym wyborem przy nierównowadze 1:9. Przy priors naturalnych (~[0.90, 0.10]) model domyślnie przypisuje bardzo małe prawdopodobieństwo klasie hazardous, co wymusza użycie ekstremalnie niskiego progu (~0.007) i generuje jeszcze więcej FP. Równe priors łagodzą ten efekt.

### Ograniczenia GaussianNB w tym zadaniu
- **Założenie normalności** — cechy są silnie skośne (power-law, outliery). GaussianNB estymuje rozkład normalny dla każdej cechy, co jest dużym uproszczeniem. Modele nieliniowe (drzewa, SVM z RBF) nie mają tego ograniczenia.
- **Założenie niezależności cech** — `est_diameter_min` i `absolute_magnitude` są fizycznie powiązane (większy obiekt = wyższa jasność absolutna). Naruszenie tego założenia może powodować zbyt skrajne prawdopodobieństwa (efekt znany jako "probability calibration problem" w Naive Bayes).
- **Wnioski porównawcze** — spodziewamy się, że modele z zespołu (drzewa decyzyjne, SVM, regresja logistyczna) osiągną porównywalny Recall przy lepszej Precision, ponieważ nie mają powyższych ograniczeń i potrafią lepiej modelować nieliniowe granice decyzyjne w tym zbiorze.